### RAG Piplines- Data Ingestion to vistore DB Piplines

In [1]:
# import libraries

import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/var/folders/40/ztqgr5z1695f_kbbl1jww6880000gn/T/ipykernel_39068/734374012.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
/Users/samjordan/Desktop/RAG_PRO/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the documents in the directory 
def load_all_pdfs(pdf_diroctory):
    """ process all pdf file in the dir """
    all_docs = []
    pdf_path = Path(pdf_diroctory)  # look for the diroctory in the project root 

    pdf_files = list(pdf_path.glob('**/*.pdf')) # look for every pdf inside the pdf_path folder
    print(f'total files : {len(pdf_files)}')

    for file in pdf_files:
        try:
            loader = PyPDFLoader(str(file))
            documents = loader.load()
            ## add source information from metadata
            for doc in documents:
                doc.metadata['source_file'] = file.name
                doc.metadata['file_type'] = 'pdf'

            # load thos rpcuments into the add_docs 
            all_docs.extend(documents)
            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f'Error loading the files:  {e}')
        
    return all_docs



pdf_load = load_all_pdfs('../data')



total files : 2
Loaded 1 pages
Loaded 56 pages


In [6]:
### Read all the pdf files in the data folder

def process_all_pdfs(pdf_directory):
    """Process all pdf files in a directory"""
    all_docs =[]
    pdf_dir = Path(pdf_directory)

    #find all PDF files 
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file}")
        
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # add the source information from metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata['file_type'] ='pdf'
            
            #load the documents into the all_docs list
            all_docs.extend(documents)
            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"error: {e}")

    print(f"Total documents processed: {len(all_docs)}")
    return all_docs

all_pdfs_documents = process_all_pdfs("../data")


found 2 PDF files to process

Processing: ../data/pdf/Sam_Aldehayyat_Resume.pdf
Loaded 1 pages

Processing: ../data/pdf/LLM_Interview_Questions_1748781586.pdf
Loaded 56 pages
Total documents processed: 57


In [7]:
all_pdfs_documents

[Document(metadata={'producer': 'Skia/PDF m150 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Sam_Aldehayyat_Resume', 'source': '../data/pdf/Sam_Aldehayyat_Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Sam_Aldehayyat_Resume.pdf', 'file_type': 'pdf'}, page_content="Sam  Aldehayyat\n   803-297-5168   |   aldehayyatsam@gmail.com   |   linkedin.com/in/sam-aldehayyat   |   github.com/Sam962  \nSUMMARY  AI  developer  with  hands-on  experience  building  LangChain  and  LangGraph  agents,  RAG  pipelines,  and  production  Python  \nservices\n \nin\n \nfintech\n \nenvironments.\n \nComfortable\n \ndesigning\n \nand\n \nshipping\n \nmulti-agent\n \nworkflows\n \nend\n \nto\n \nend\n \n–\n \nfrom\n \nAPI\n \nintegration\n \nand\n \ncontainerized\n \ndeployment\n \nto\n \nobservability\n \nand\n \naccess\n \ncontrol\n \nin\n \nsecure\n \nenterprise\n \nsettings.\n \nTECHNICAL  SKILLS  Languages:   Python,  JavaScript,  TypeScript,  SQL, 

### STEP2: SPLIT AND CHUNK

In [8]:
from pydoc import doc


def split_doc(documents, chunk_size = 500 , chunk_overlap = 200):
    """ Split Documents into chunks """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators= ['\n\n', '\n', ',' ' ', '']
    )
    # chunk the documenets 
    split_doc = text_splitter.split_documents(documents)
    print(f"Found {len(documents)} documents split into: {len(split_doc)} chunks")

    if split_doc:
        print(f"\n chunks Example")
        print(f" Content:  {split_doc[0].page_content[:200]} .. ")
        print(f" Metadata: {split_doc[0].metadata}")

    return split_doc




In [9]:
chunks = split_doc(all_pdfs_documents)

Found 57 documents split into: 129 chunks

 chunks Example
 Content:  Sam  Aldehayyat
   803-297-5168   |   aldehayyatsam@gmail.com   |   linkedin.com/in/sam-aldehayyat   |   github.com/Sam962  
SUMMARY  AI  developer  with  hands-on  experience  building  LangChain  an .. 
 Metadata: {'producer': 'Skia/PDF m150 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Sam_Aldehayyat_Resume', 'source': '../data/pdf/Sam_Aldehayyat_Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Sam_Aldehayyat_Resume.pdf', 'file_type': 'pdf'}


In [11]:
### Text splitting get tinto 

def split_documents(documents, chunk_size = 512, chunk_overlap=200):
    """split documents into chuncks """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function = len,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    # chunk the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # show example of a chunk
    if split_docs:
        print(f'\nExample of a chunk:')
        print(f"Content: {split_docs[0].page_content[:200]} ...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs


In [12]:
chunks = split_documents(all_pdfs_documents)

Split 57 documents into 127 chunks

Example of a chunk:
Content: Sam  Aldehayyat
   803-297-5168   |   aldehayyatsam@gmail.com   |   linkedin.com/in/sam-aldehayyat   |   github.com/Sam962  
SUMMARY  AI  developer  with  hands-on  experience  building  LangChain  an ...
Metadata: {'producer': 'Skia/PDF m150 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Sam_Aldehayyat_Resume', 'source': '../data/pdf/Sam_Aldehayyat_Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Sam_Aldehayyat_Resume.pdf', 'file_type': 'pdf'}


### Step 3: Ebbidding AND vectoreStore DB..


In [14]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings   
import uuid # giving an ID number so each chunk has its own unique number 
from typing import List, Tuple, Dict, Any
from sklearn.metrics.pairwise import cosine_similarity 


In [15]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer    """
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Inetialize the embedding manager

        args: 
        model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """ Load the SentenceTransformer model """
        try: 
            print(f"Loading embedding model : {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded succssefully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise


    def generate_embedding(self, texts: List[str]) -> np.ndarray:
        """
        Generate embedding for a list of texts

        args:
            texts: list of text strings to embed

        Returnes:
            numpy array of embeddings with shape (len(texts), embedding_dim) 
        """        
        if not self.model:
            raise ValueError("model not loaded")

        print(f"Generateing embedding for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generatted embedding with shape: {embeddings.shape}")

        return embeddings


### Initiatlize the embedding manager

embedding_manager = EmbeddingManager()

embedding_manager
        



Loading embedding model : all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8589.76it/s]


Model loaded succssefully. Embedding dimension: 384


### Step4:  VectoreStore db


In [16]:
class VectorStore:
    """ Mnages documents embedding in chromadb vector Store """

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        args: 
        collection_name: name of chromadb collection 
        persist_d
        
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client =None
        self.collection =None
        self._intialize_store()

    def _intialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # create persistent ChromaDB client 
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name =self.collection_name,
                metadata={"description": "pdf document embedding for RAG"}
            )
            print(f"Vector store initialize. collection: {self.collection_name}")
            print(f"Existing document in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.array):
        """
        Add documents and thier embedding to the vector store

        args: 
        documents:" List of langChain documents 
        embeddings: Corresponding embedding for the documents 
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings!")

        print(f"Adding {len(documents)} documents to vector store...")

        #prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list =[]

        for i, (doc,embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            #embedding 
            embeddings_list.append(embedding.tolist())


        # Add to Colection
        try:
            self.collection.add(
                ids = ids,
                embeddings = embeddings_list,
                metadatas = metadatas,
                documents = documents_text,

            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents tp vectore store: {e}")
            raise


vectorstore = VectorStore()
vectorstore




    
    

Vector store initialize. collection: pdf_documents
Existing document in collection: 126


In [17]:
### convert the text to embeddings

texts = [doc.page_content for doc in chunks]


## Generate the Embeddings

embeddings = embedding_manager.generate_embedding(texts)

## store in the vector database

vectorstore.add_documents(chunks, embeddings)

Generateing embedding for 127 texts...


Batches: 100%|██████████| 4/4 [00:02<00:00,  1.55it/s]

Generatted embedding with shape: (127, 384)
Adding 127 documents to vector store...
Successfully added 127 documents to vector store
Total documents in collection: 253


### Retriever Pipeline From VectorStore

In [18]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        args:
            vector_store: vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings 
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retriever(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """ 
        Retrieve relevant documents for aquery 

        args:
            query: The search query
            top_k: Number of top results to return 
            score_threashold: Minimum Similarity score threashold

        Returns:
            List of dictionaries containing retrieved documnets and metadata
        """   

        print(f"Retrieving documents for a quesry: '{query}'")
        print(f"Top k : {top_k}, score threshold: {score_threshold}")

        # Generate query embedding 
        query_embedding = self.embedding_manager.generate_embedding([query])[0]

        # search in tthe vector store
        
        try:
            results = self.vector_store.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results= top_k
            )

            #process results 
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distance = results['distances'][0]
                ids = results['ids'][0]


                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distance)):
                    # convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i +1
                        })

                print(f"Retrieved {len(retrieved_docs)} documemts (after the filtering)")
            
            else: 
                print("No documents founded!!")
            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vectorstore, embedding_manager)

rag_retriever

In [19]:
rag_retriever

In [20]:
# ask query #here :  
rag_retriever.retriever(" What is beam search, and how does it differ from greedy")

Retrieving documents for a quesry: ' What is beam search, and how does it differ from greedy'
Top k : 5, score threshold: 0.0
Generateing embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.28s/it]

Generatted embedding with shape: (1, 384)
Retrieved 4 documemts (after the filtering)


[{'id': 'doc_768fc33a_19',
  'content': 'Bhavishya Pandit\nQ3. What is beam search, and how does it differ from greedy\ndecoding?\nAns - Beam search is a search algorithm used during text generation\nto find the most likely sequence of words. Instead of choosing the\nsingle highest-probability word at each step (as greedy decoding\ndoes), beam search explores multiple possible sequences in parallel,\nmaintaining a set of the top k candidates (beams). It balances\nbetween finding high-probability sequences and exploring',
  'metadata': {'producer': 'PyPDF',
   'file_type': 'pdf',
   'source': '../data/pdf/LLM_Interview_Questions_1748781586.pdf',
   'content_length': 475,
   'source_file': 'LLM_Interview_Questions_1748781586.pdf',
   'total_pages': 56,
   'doc_index': 19,
   'creator': 'PyPDF',
   'page': 4,
   'page_label': '5',
   'creationdate': ''},
  'similarity_score': 0.6862803101539612,
  'distance': 0.3137196898460388,
  'rank': 1},
 {'id': 'doc_feb9546d_20',
  'content': 'Bhavi

## Integration Vectordb Context pipline with LLM output

In [22]:
### Simple RAG pipline with groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the groq LLM (set your GROQ_API_KEY in the env)

groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="llama-3.1-8b-instant",
    temperature=0.1,
    max_tokens=1024,
)

## 2. Simple RAG function: retrive context + generate response

def rag_simple(query, retriever, llm, top_k = 3):
    ## retrieve the context 
    results= retriever.retriever(query, top_k= top_k)
    context= "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question!!"

    # generate the answer using GROK LLM 
    prompt = f""" Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer: 
    """
    response= llm.invoke(prompt.format(context= context, query = query))
    return response.content



In [23]:
answer = rag_simple("What is beam search, and how does it differ from greedy", rag_retriever, llm)
print(answer)

Retrieving documents for a quesry: 'What is beam search, and how does it differ from greedy'
Top k : 3, score threshold: 0.0
Generateing embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 20.31it/s]

Generatted embedding with shape: (1, 384)
Retrieved 3 documemts (after the filtering)


NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.1-8b-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

## Enhance RAG Pipeline Features

In [24]:
def rag_advance(query, retriever, llm , top_k=5, min_score = 0.2 , return_context = False):
    """
    RAG Pipline with extra Features:
    -   Return answer, sources, confident score, and optionally full context.
    """
    results = retriever.retriever(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}

    # Prepare context and sources 
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '....'
    }for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # Genrate answer
    prompt= f"""Use the following context to answer the question concisely
        context:
        {context}
        Question: {query}

        Answer:"""
    response = llm.invoke(prompt)
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }

    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advance(input("What is your Question?"), rag_retriever, llm, top_k= 3, return_context=True)
print("Answer", result['answer'])
print('Sources', result['sources'])
print('Confidence', result['confidence'])
print('Context Preview:', result['context'][:300])



Retrieving documents for a quesry: ''
Top k : 3, score threshold: 0.2
Generateing embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.78it/s]

Generatted embedding with shape: (1, 384)
Retrieved 0 documemts (after the filtering)
Answer No relevant context found.
Sources []
Confidence 0.0
Context Preview: 


## Advance RAG

In [25]:
### Advance RAG Pipeline: Streaming, Citations , History, Summerization 

from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  #store query History 

    def query(self, question: str, top_k = 5, min_score: float = 0.2, stream: bool = False, summerize: bool = True) -> Dict[str, Any]:
        # Retrieve relevant documents
        results =  self.retriever.retriever(question, top_k=top_k, score_threshold= min_score)
        if not results: 
            answer = "No relevant Context found. "
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{ 
                'source' : doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page' : doc['metadata'].get('page', 'unknown'),
                'score' : doc['similarity_score'],
                'preview': doc['content'][:120] + '....'
            }for doc in results]
            # streaming answer simulation 
            prompt = f""" Use the following context to answer the question cocisely.
            context: 
            {context}
            
            Question: {question}

            answer:          
            """
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content
        # Add Citations to answer
        citations = [f"[{i+1}] {src['source']} (page{src['page']}" for i , src in enumerate(sources)]
        answer_with_citation = answer + "\n\nCitation: \n" + "\n".join(citations) if citations else answer

        # Optionally summerize answer

        summary = None
        if summerize and answer:
            summary_prompt = f"Summarize the following answer into 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content 

        # store query history 
        self.history.append({
            'question' : question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })
        return {

            'question': question, 
            'answer': answer_with_citation,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query(input(" write your query ..."), top_k=3, min_score=0.3, stream=True, summerize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])




Retrieving documents for a quesry: ''
Top k : 3, score threshold: 0.3
Generateing embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.47it/s]

Generatted embedding with shape: (1, 384)
Retrieved 0 documemts (after the filtering)


NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.1-8b-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}